# Laboratorio #6 - Aprendizaje por Refuerzo


- Francis Aguilar, 22243
- Jose Marchena, 22398
- Cesar Lopez, 22535


# Instrucciones

Una empresa de robótica de asistencia médica está desarrollando un exoesqueleto para rehabilitación de movilidad en pacientes con lesiones de rodilla. El sistema de control debe aprender a asistir el movimiento de la pierna del paciente de forma suave y eficiente, minimizando el esfuerzo del motor y maximizando la fluidez del movimiento. Antes de trabajar con el hardware real, el equipo de ingeniería necesita validar si los métodos de gradiente de política son apropiados para este dominio de control continuo, usando entornos de simulación estándar como proxy del problema real. Su grupo ha sido contratado para implementar y comparar REINFORCE con línea base y un Actor-Critic simple, analizar sus propiedades de convergencia, e investigar el estado del arte en control de exoesqueletos con RL para producir un dictamen técnico fundamentado.


## Task 1 (Entrega Parcial)

Respondan las siguientes preguntas con argumentación técnica rigurosa antes de implementar nada:


1. El entorno de simulación que usarán es LunarLanderContinuous-v2 de Gymnasium, que tiene un espacio de acción continuo de dos dimensiones representando la fuerza de dos propulsores. Argumenten formalmente por qué Q-Learning tabular y DQN son inapropiados para este entorno. Su argumento debe mencionar explícitamente el espacio de acción, el operador argmax, y la representación de la política.


El espacio de acciones es continuo: $\mathcal{A}=[-1,1]^2$, donde cada componente controla un propulsor. Por tanto, hay infinitas acciones posibles para cada estado. Q-Learning tabular requiere almacenar un valor separado para cada par $(s,a)$ y actualizar $Q(s,a)$; esto no es finito ni práctico cuando $a\in[-1,1]^2$ (y, además, el estado de LunarLander también es continuo). Una discretización de las acciones produciría otro problema finito, pero cambiaría el entorno: solo se podrían ejecutar los pocos pares discretizados y se perdería precisión en el control.

DQN usa una red $Q_\theta(s,a)$, pero la política codiciosa que necesita es $\pi(s)=\arg\max_{a\in\mathcal{A}}Q_\theta(s,a)$. En un espacio discreto, el argmax se obtiene enumerando las acciones que salen de la red. En este caso no se pueden enumerar infinitas acciones; hallar el máximo requeriría resolver una optimización continua en cada paso, y no corresponde a la salida usual de un DQN, que produce un valor por acción discreta. Discretizar para aplicar DQN hereda la aproximación anterior y puede generar acciones bruscas o subóptimas.

Finalmente, una política DQN es esencialmente categórica: selecciona una acción de un conjunto finito (con exploración $\epsilon$-greedy). LunarLanderContinuous-v2 necesita representar directamente un vector real de dos componentes y explorar alrededor de él. Una política estocástica continua, por ejemplo una Gaussiana parametrizada por una red, representa naturalmente $\pi_\theta(a\mid s)$ y permite aplicar gradientes de política. Por estas razones, Q-Learning tabular y DQN no son adecuados sin una discretización artificial; métodos actor-critic o REINFORCE con política continua sí lo son.


2. Para LunarLanderContinuous-v2, la política se parametrizará como una distribución Gaussiana 𝜋𝜃(𝑎 ∣ 𝑠) = 𝒩(𝜇𝜃, (𝑠)𝜎2𝐼) donde 𝜇𝜃(𝑠) es la salida de una red neuronal. Expliquen cómo se calcula ∇𝜃 ln 𝜋𝜃 (𝐴𝑡 ∣ 𝑆𝑡) para esta parametrización específica. Desarrollen la expresión analítica del
   gradiente del logaritmo de la densidad Gaussiana respecto a 𝜃, identificando qué parte depende de 𝜃 y qué parte no


Sea $d=2$, $\mu_\theta=\mu_\theta(s)$ y supongamos que $\sigma$ es constante e independiente de $\theta$. La densidad es $\pi_\theta(a\mid s)=(2\pi\sigma^2)^{-d/2}\exp[-\|a-\mu_\theta(s)\|^2/(2\sigma^2)]$. Su logaritmo es

$$\log\pi_\theta(a\mid s)=-\frac d2\log(2\pi\sigma^2)-\frac{1}{2\sigma^2}\|a-\mu_\theta(s)\|^2.$$

Al derivar respecto a $\theta$, el primer término desaparece porque $\sigma$ y $d$ no dependen de los parámetros. Aplicando la regla de la cadena:$$\nabla_\theta\log\pi_\theta(a\mid s)=\frac{1}{\sigma^2}(a-\mu_\theta(s))^\top\nabla_\theta\mu_\theta(s).$$

Para la muestra observada $(S_t,A_t)$:$$\nabla_\theta\log\pi_\theta(A_t\mid S_t)=\frac{A_t-\mu_\theta(S_t)}{\sigma^2}\,\nabla_\theta\mu_\theta(S_t).$$

En implementación, $(A_t-\mu_\theta)/\sigma^2$ es el error normalizado y $\nabla_\theta\mu_\theta$ se obtiene mediante backpropagation. La acción $A_t$ y el estado $S_t$ son datos de la trayectoria, no parámetros; $\sigma$ tampoco aporta gradiente bajo esta suposición. Si $\sigma=\sigma_\theta(s)$ fuera aprendible, también aparecería el término $[(a-\mu)^2/\sigma^2-1]\nabla_\theta\log\sigma$ por dimensión.


3. Comparen formalmente REINFORCE con línea base y Actor-Critic en términos de sesgo y varianza del estimador del gradiente. Para cada algoritmo identifiquen: qué usa como estimador de la ventaja 𝐴̂ 𝑡, qué componente introduce sesgo, y qué componente introduce varianza. Predigan cuál algoritmo esperan que converja más rápido en LunarLanderContinuous-v2 y justifiquen esa predicción.


El gradiente de política puede escribirse como $g=\mathbb{E}[\nabla_\theta\log\pi_\theta(A_t\mid S_t)A^{\pi}(S_t,A_t)]$. Una línea base que depende solo de $S_t$ no cambia su esperanza porque $\mathbb{E}_{A\sim\pi}[\nabla_\theta\log\pi(A\mid s)]=0$.

| Algoritmo | Estimador de $\hat A_t$ | Sesgo | Varianza |
|---|---|---|---|
| REINFORCE con línea base | $\hat A_t=G_t-b_\phi(S_t)$, usualmente $b_\phi\approx V^\pi(S_t)$ | Si $b$ depende solo de $s$, ninguno en el gradiente por la línea base. Un baseline aprendido imperfecto no introduce sesgo, aunque sí puede no reducir bien la varianza. El retorno Monte Carlo es insesgado (bajo una trayectoria y política correctas). | Alta: $G_t$ incluye toda la aleatoriedad futura; también hay ruido de las acciones y de la terminación de episodios. |
| Actor-Critic simple | $\hat A_t=R_{t+1}+\gamma V_\phi(S_{t+1})-V_\phi(S_t)$ (TD, o una variante multi-paso/GAE) | El crítico aproximado y el bootstrap $V_\phi(S_{t+1})$ pueden hacer $\hat A_t$ sesgado si $V_\phi\ne V^\pi$. Además, durante el aprendizaje el objetivo cambia. | Menor que Monte Carlo porque usa un paso y elimina gran parte del retorno futuro, pero conserva ruido de transición, recompensa y acción; el aprendizaje del crítico añade error de aproximación. |
|

En REINFORCE, el estimador completo sigue siendo insesgado si $b(S_t)$ se considera independiente de la acción usada en el término de actor. En Actor-Critic, el sesgo por bootstrap es el intercambio por una varianza menor y actualizaciones más frecuentes. En LunarLanderContinuous-v2 esperamos que Actor-Critic converja más rápido: sus actualizaciones son por paso, aprovechan mejor los datos y la señal TD tiene menor varianza, algo importante en episodios largos y con recompensas escasas. REINFORCE puede ser más estable conceptualmente por su estimador Monte Carlo, pero normalmente necesita más episodios para que el promedio de retornos reduzca el ruido. La predicción supone un crítico suficientemente bien entrenado; al inicio, un crítico muy malo puede hacer que Actor-Critic sea temporalmente inestable.


4. El entorno de exoesqueleto real tiene una restricción que -v2 no tiene:
   las acciones deben ser suaves en el tiempo para no causar movimientos bruscos que dañen al paciente. Argumenten cómo modificarían la función de recompensa y la parametrización de la política para incorporar esa restricción. ¿Cambiaría eso la elección entre REINFORCE y Actor-Critic?


La recompensa debe penalizar explícitamente el cambio entre acciones consecutivas. Una forma es

$$r'_t=r_t-\lambda_\Delta\|a_t-a_{t-1}\|^2-\lambda_u\|a_t\|^2,$$

donde el primer término suaviza la trayectoria y el segundo limita el esfuerzo del motor. Para una restricción de seguridad estricta, no basta una penalización: se debe saturar la acción, imponer $\|a_t-a_{t-1}\|\leq\Delta_{max}$, o usar un filtro/safety layer que proyecte la acción propuesta al conjunto permitido. El estado debe incluir $a_{t-1}$ (o una historia reciente), para que la penalización sea Markoviana. $\lambda_\Delta$ debe calibrarse junto con el desempeño de rehabilitación: demasiado grande puede impedir asistir al paciente.

La política también puede generar incrementos suaves en vez de acciones independientes: $a_t=\operatorname{clip}(a_{t-1}+\Delta a_t,-1,1)$, con $\Delta a_t$ producido por una Gaussiana de media neuronal y acotado por $\Delta_{max}$. Alternativamente, puede usarse una política Gaussiana con media condicionada en $a_{t-1}$ y covarianza pequeña, o un proceso de ruido temporalmente correlacionado. Para seguridad clínica, conviene combinar esta parametrización con límites físicos, control de tasa de cambio y una política de respaldo.

La modificación no cambia la elección fundamental entre REINFORCE y Actor-Critic: ambos pueden optimizar la recompensa modificada y usar una política continua. Sí cambia la información necesaria (acción previa) y puede aumentar la importancia de una señal de aprendizaje eficiente. Por ello seguiríamos prefiriendo Actor-Critic en este entorno: el término de suavidad puede hacer la recompensa más densa, mientras que TD reduce la varianza y permite corregir rápidamente acciones inseguras. REINFORCE sigue siendo válido y ofrece un estimador Monte Carlo menos sesgado, pero suele requerir más episodios; ninguna de las dos alternativas sustituye las restricciones de seguridad explícitas.


# Task 2


Implementen REINFORCE con línea base y Actor-Critic sobre LunarLanderContinuous-v2 con las
siguientes especificaciones. 

Ambos algoritmos deben usar una red neuronal con dos capas ocultas de 64 neuronas y activaciones ReLU
para parametrizar la política. La media 𝜇𝜃(𝑠) es la salida de la red con activación tanh para restringir las acciones al rango válido. La desviación estándar 𝜎 debe ser un parámetro aprendible independiente del estado, inicializado en 0.5. Para REINFORCE, la línea base debe ser una red separada que aproxime 𝑉𝜋(𝑠), entrenada con pérdida cuadrática sobre los retornos observados. Para Actor-Critic, el Critic comparte el cuerpo de la red con el Actor y tiene una cabeza de salida escalar para 𝑉̂𝑤 (𝑠). La implementación debe incluir:


- Ambos algoritmos implementados desde cero usando PyTorch para la diferenciación automática. No
se permite usar implementaciones preconstruidas de REINFORCE o Actor-Critic de ninguna librería
de RL.
- Registro por episodio de recompensa total, norma del gradiente del Actor ∥ ∇𝜃𝐿 ∥, y entropía de la
política 𝑆[𝜋𝜃].
- Entrenamiento de ambos algoritmos durante al menos 1000 episodios con la misma semilla
aleatoria para comparación justa.
- Cuatro gráficas: curvas de aprendizaje de ambos algoritmos en la misma figura con media móvil de
20 episodios, evolución de la norma del gradiente, evolución de la entropía de la política, y una
visualización de al menos un episodio de la política aprendida mediante env.render().

## Configuración e imports

Implementación desde cero (sin usar REINFORCE/Actor-Critic preconstruidos de librerías de RL). Se usa `gymnasium` para el entorno y `PyTorch` únicamente para la diferenciación automática.

Dependencias necesarias (ejecutar una sola vez si hace falta):
```
pip install gymnasium[box2d] torch matplotlib numpy
```

In [ ]:
import random
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal

from matplotlib import animation
from IPython.display import HTML

# LunarLanderContinuous-v2 fue renombrado a -v3 en versiones recientes de Gymnasium.
try:
    gym.make("LunarLanderContinuous-v2")
    ENV_ID = "LunarLanderContinuous-v2"
except Exception:
    ENV_ID = "LunarLanderContinuous-v3"

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HIDDEN_SIZE = 64
INIT_STD = 0.5
GAMMA = 0.99
EPISODES = 1000
MAX_STEPS = 1000

print(f"Entorno: {ENV_ID} | device: {DEVICE}")


### Redes neuronales

Ambos algoritmos usan una política Gaussiana $\pi_\theta(a\mid s)=\mathcal N(\mu_\theta(s),\sigma^2 I)$: dos capas ocultas de 64 neuronas con ReLU, media con activación `tanh` (acota la salida a $[-1,1]$) y $\log\sigma$ como parámetro aprendible independiente del estado, inicializado en $\sigma=0.5$.

- `GaussianPolicyNet`: red de política independiente, usada como Actor de REINFORCE.
- `ValueNet`: red separada que aproxima $V^\pi(s)$, usada como línea base de REINFORCE.
- `ActorCriticNet`: cuerpo compartido entre Actor y Critic, con una cabeza para $\mu_\theta(s)$ (+ $\log\sigma$) y una cabeza escalar para $\hat V_w(s)$.

In [ ]:
class GaussianPolicyNet(nn.Module):
    """Actor de REINFORCE: mu_theta(s) con tanh, log_std aprendible independiente de s."""

    def __init__(self, obs_dim, act_dim, hidden=HIDDEN_SIZE, init_std=INIT_STD):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.log(torch.full((act_dim,), init_std)))

    def forward(self, state):
        h = self.body(state)
        mu = torch.tanh(self.mu_head(h))
        std = torch.exp(self.log_std).expand_as(mu)
        return mu, std


class ValueNet(nn.Module):
    """Linea base V_phi(s) para REINFORCE, entrenada con perdida cuadratica sobre G_t."""

    def __init__(self, obs_dim, hidden=HIDDEN_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, state):
        return self.net(state).squeeze(-1)


class ActorCriticNet(nn.Module):
    """Actor-Critic con cuerpo compartido: cabeza de politica (mu, log_std) y cabeza de valor V_w(s)."""

    def __init__(self, obs_dim, act_dim, hidden=HIDDEN_SIZE, init_std=INIT_STD):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden, act_dim)
        self.log_std = nn.Parameter(torch.log(torch.full((act_dim,), init_std)))
        self.v_head = nn.Linear(hidden, 1)

    def forward(self, state):
        h = self.body(state)
        mu = torch.tanh(self.mu_head(h))
        std = torch.exp(self.log_std).expand_as(mu)
        value = self.v_head(h).squeeze(-1)
        return mu, std, value


def grad_norm(parameters):
    """Norma L2 total del gradiente sobre un conjunto de parametros (||grad_theta L||)."""
    total_sq = 0.0
    for p in parameters:
        if p.grad is not None:
            total_sq += p.grad.data.norm(2).item() ** 2
    return total_sq ** 0.5


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


### REINFORCE con línea base

Por episodio: se recolecta la trayectoria completa, se calculan los retornos Monte Carlo $G_t=\sum_{k\ge t}\gamma^{k-t}r_k$, la ventaja $\hat A_t=G_t-V_\phi(S_t)$ (con $V_\phi$ tratada como constante para el gradiente del actor) y se actualiza el actor con $\nabla_\theta\sum_t \log\pi_\theta(A_t\mid S_t)\hat A_t$. La línea base $V_\phi$ se entrena por separado con pérdida cuadrática $\mathrm{MSE}(V_\phi(S_t),G_t)$.

In [ ]:
def train_reinforce(env_id, episodes=EPISODES, gamma=GAMMA, lr_policy=3e-4, lr_value=1e-3,
                     seed=SEED, max_steps=MAX_STEPS, device=DEVICE, log_every=50):
    set_seed(seed)
    env = gym.make(env_id)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    policy = GaussianPolicyNet(obs_dim, act_dim).to(device)
    value_net = ValueNet(obs_dim).to(device)
    policy_opt = optim.Adam(policy.parameters(), lr=lr_policy)
    value_opt = optim.Adam(value_net.parameters(), lr=lr_value)

    logs = defaultdict(list)

    for ep in range(episodes):
        state, _ = env.reset(seed=seed + ep)
        states, log_probs, rewards, entropies = [], [], [], []
        done = False
        steps = 0

        while not done and steps < max_steps:
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
            mu, std = policy(state_t)
            dist = Normal(mu, std)
            action = dist.sample()

            next_state, reward, terminated, truncated, _ = env.step(
                torch.clamp(action, -1.0, 1.0).cpu().numpy()
            )
            done = terminated or truncated

            states.append(state_t)
            log_probs.append(dist.log_prob(action).sum())
            rewards.append(reward)
            entropies.append(dist.entropy().sum())

            state = next_state
            steps += 1

        # Retornos Monte Carlo G_t (descontados, de atras hacia adelante)
        returns = []
        G = 0.0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        states_t = torch.stack(states)
        values = value_net(states_t)
        advantages = returns - values.detach()

        log_probs_t = torch.stack(log_probs)
        policy_loss = -(log_probs_t * advantages).sum()

        policy_opt.zero_grad()
        policy_loss.backward()
        g_norm = grad_norm(policy.parameters())
        policy_opt.step()

        value_loss = F.mse_loss(values, returns)
        value_opt.zero_grad()
        value_loss.backward()
        value_opt.step()

        logs["rewards"].append(sum(rewards))
        logs["grad_norms"].append(g_norm)
        logs["entropies"].append(torch.stack(entropies).mean().item())

        if (ep + 1) % log_every == 0:
            recent = np.mean(logs["rewards"][-log_every:])
            print(f"[REINFORCE] episodio {ep + 1}/{episodes} | recompensa media (ult. {log_every}): {recent:.2f}")

    env.close()
    return policy, value_net, logs


### Actor-Critic (TD(0), un paso)

Actualización en cada paso (no al final del episodio): con el error TD $\delta_t=r_t+\gamma \hat V_w(S_{t+1})(1-\text{done})-\hat V_w(S_t)$ como estimador de la ventaja, la pérdida del actor es $-\log\pi_\theta(A_t\mid S_t)\,\delta_t$ (con $\delta_t$ tratado como constante) y la del crítico $\delta_t^2$. Como el cuerpo de la red es compartido, ambas pérdidas se combinan en un único `loss` y se retropropagan juntas; la norma de gradiente registrada es la de toda la red (actor+cuerpo compartido), ya que ese es el gradiente que efectivamente mueve la política.

In [ ]:
def train_actor_critic(env_id, episodes=EPISODES, gamma=GAMMA, lr=3e-4, value_coef=0.5,
                        seed=SEED, max_steps=MAX_STEPS, device=DEVICE, log_every=50):
    set_seed(seed)
    env = gym.make(env_id)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    net = ActorCriticNet(obs_dim, act_dim).to(device)
    optimizer = optim.Adam(net.parameters(), lr=lr)

    logs = defaultdict(list)

    for ep in range(episodes):
        state, _ = env.reset(seed=seed + ep)
        done = False
        steps = 0
        ep_reward = 0.0
        ep_grad_norms, ep_entropies = [], []

        while not done and steps < max_steps:
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
            mu, std, value = net(state_t)
            dist = Normal(mu, std)
            action = dist.sample()
            log_prob = dist.log_prob(action).sum()
            entropy = dist.entropy().sum()

            next_state, reward, terminated, truncated, _ = env.step(
                torch.clamp(action, -1.0, 1.0).cpu().numpy()
            )
            done = terminated or truncated

            with torch.no_grad():
                next_state_t = torch.as_tensor(next_state, dtype=torch.float32, device=device)
                _, _, next_value = net(next_state_t)
                td_target = reward + gamma * next_value * (0.0 if done else 1.0)
            td_error = td_target - value

            actor_loss = -log_prob * td_error.detach()
            critic_loss = td_error.pow(2)
            loss = actor_loss + value_coef * critic_loss

            optimizer.zero_grad()
            loss.backward()
            ep_grad_norms.append(grad_norm(net.parameters()))
            optimizer.step()

            ep_reward += reward
            ep_entropies.append(entropy.item())

            state = next_state
            steps += 1

        logs["rewards"].append(ep_reward)
        logs["grad_norms"].append(float(np.mean(ep_grad_norms)))
        logs["entropies"].append(float(np.mean(ep_entropies)))

        if (ep + 1) % log_every == 0:
            recent = np.mean(logs["rewards"][-log_every:])
            print(f"[Actor-Critic] episodio {ep + 1}/{episodes} | recompensa media (ult. {log_every}): {recent:.2f}")

    env.close()
    return net, logs


### Entrenamiento

Se entrenan ambos algoritmos durante `EPISODES` (≥ 1000) episodios usando la misma semilla `SEED`, para que la comparación sea justa (mismas condiciones iniciales de los episodios de entrenamiento).

In [ ]:
reinforce_policy, reinforce_value_net, reinforce_logs = train_reinforce(
    ENV_ID, episodes=EPISODES, gamma=GAMMA, seed=SEED, max_steps=MAX_STEPS, device=DEVICE
)


In [ ]:
actor_critic_net, ac_logs = train_actor_critic(
    ENV_ID, episodes=EPISODES, gamma=GAMMA, seed=SEED, max_steps=MAX_STEPS, device=DEVICE
)


### Gráficas

Cuatro gráficas requeridas: (1) curvas de aprendizaje de ambos algoritmos con media móvil de 20 episodios, (2) evolución de la norma del gradiente del actor, (3) evolución de la entropía de la política, y (4) visualización de un episodio de la política aprendida con `env.render()`.

In [ ]:
def moving_average(x, window=20):
    x = np.asarray(x, dtype=np.float64)
    if len(x) < window:
        return x
    return np.convolve(x, np.ones(window) / window, mode="valid")


# --- Grafica 1: curvas de aprendizaje (recompensa) con media movil de 20 episodios ---
window = 20
plt.figure(figsize=(10, 6))
plt.plot(range(window, len(reinforce_logs["rewards"]) + 1), moving_average(reinforce_logs["rewards"], window),
         label="REINFORCE + baseline")
plt.plot(range(window, len(ac_logs["rewards"]) + 1), moving_average(ac_logs["rewards"], window),
         label="Actor-Critic")
plt.xlabel("Episodio")
plt.ylabel(f"Recompensa total (media móvil de {window} episodios)")
plt.title("Curvas de aprendizaje: REINFORCE con línea base vs. Actor-Critic")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# --- Grafica 2: evolucion de la norma del gradiente del actor ||grad_theta L|| ---
plt.figure(figsize=(10, 6))
plt.plot(reinforce_logs["grad_norms"], alpha=0.25, color="tab:blue")
plt.plot(range(window, len(reinforce_logs["grad_norms"]) + 1),
         moving_average(reinforce_logs["grad_norms"], window),
         color="tab:blue", label="REINFORCE + baseline")
plt.plot(ac_logs["grad_norms"], alpha=0.25, color="tab:orange")
plt.plot(range(window, len(ac_logs["grad_norms"]) + 1),
         moving_average(ac_logs["grad_norms"], window),
         color="tab:orange", label="Actor-Critic")
plt.xlabel("Episodio")
plt.ylabel(r"$\|\nabla_\theta L\|$")
plt.yscale("log")
plt.title("Evolución de la norma del gradiente del Actor (líneas tenues = valor crudo)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# --- Grafica 3: evolucion de la entropia de la politica S[pi_theta] ---
plt.figure(figsize=(10, 6))
plt.plot(reinforce_logs["entropies"], alpha=0.25, color="tab:blue")
plt.plot(range(window, len(reinforce_logs["entropies"]) + 1),
         moving_average(reinforce_logs["entropies"], window),
         color="tab:blue", label="REINFORCE + baseline")
plt.plot(ac_logs["entropies"], alpha=0.25, color="tab:orange")
plt.plot(range(window, len(ac_logs["entropies"]) + 1),
         moving_average(ac_logs["entropies"], window),
         color="tab:orange", label="Actor-Critic")
plt.xlabel("Episodio")
plt.ylabel(r"Entropía $S[\pi_\theta]$ (promedio por episodio)")
plt.title("Evolución de la entropía de la política (líneas tenues = valor crudo)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# --- Grafica 4: visualizacion de un episodio de la politica aprendida via env.render() ---

def make_policy_fn(net, is_actor_critic, device=DEVICE, deterministic=True):
    """Envuelve una red entrenada en una funcion state -> accion (usa la media, sin exploracion)."""

    def policy_fn(state):
        state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
        with torch.no_grad():
            if is_actor_critic:
                mu, std, _ = net(state_t)
            else:
                mu, std = net(state_t)
            action = mu if deterministic else Normal(mu, std).sample()
        return torch.clamp(action, -1.0, 1.0).cpu().numpy()

    return policy_fn


def run_episode_render(env_id, policy_fn, max_steps=MAX_STEPS, seed=SEED):
    env = gym.make(env_id, render_mode="rgb_array")
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0
    done = False
    steps = 0
    while not done and steps < max_steps:
        action = policy_fn(state)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
        frames.append(env.render())
        steps += 1
    env.close()
    return frames, total_reward


def animate_frames(frames, interval=30):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")
    im = plt.imshow(frames[0])

    def update(i):
        im.set_data(frames[i])
        return [im]

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=True)
    plt.close(fig)
    return anim


# Se visualiza la politica final de Actor-Critic (cambiar a reinforce_policy para ver la otra)
frames, total_reward = run_episode_render(
    ENV_ID, make_policy_fn(actor_critic_net, is_actor_critic=True), seed=SEED
)
print(f"Recompensa total del episodio visualizado: {total_reward:.2f}")
anim = animate_frames(frames)
HTML(anim.to_jshtml())
